# NZZ ContextAI — Explorer Notebook
Datenexploration und Pipeline-Test mit den NZZ JSON-Daten.

## 1. Setup

In [ ]:
import sys
sys.path.append('../src')

import json
import glob
import pandas as pd
import matplotlib.pyplot as plt

from config import NZZ_JSON_GLOB, EMBEDDING_MODEL, CHROMA_PATH, CHROMA_COLLECTION, USE_RERANKING
from preprocess import load_dataset, preprocess, strip_html
from chunking import chunk_dataframe

print("Setup abgeschlossen.")
print(f"NZZ JSON Glob: {NZZ_JSON_GLOB}")

## 2. Rohdaten-Struktur
Einzelne JSON-Datei inspizieren — Felder und Datenqualität verstehen.

In [ ]:
# Eine Datei laden und Wrapper-Struktur zeigen
sample_file = sorted(glob.glob(NZZ_JSON_GLOB))[-1]  # neueste Datei
with open(sample_file) as f:
    raw = json.load(f)

# Struktur: entweder [{...}] oder {...}
wrapper = raw[0] if isinstance(raw, list) else raw
result  = wrapper.get("result", [])

print(f"Datei:        {sample_file.split('/')[-1]}")
print(f"searchTime:   {wrapper.get('searchTime')} ms")
print(f"numFound:     {wrapper.get('numFound')} Artikel (gesamt in dieser Suche)")
print(f"result-Länge: {len(result)} Artikel in dieser Datei")

article = result[1]   # Index 1 = erster Volltextartikel (0 oft Multimedia)
print("\n── Felder eines Artikels ──")
for key, val in article.items():
    v = str(val)
    print(f"  {key:<30} {v[:80]}")

In [ ]:
# Schlüsselfelder eines Artikels im Klartext
print("TITEL:  ", article.get("ueberschrift_ctx", ""))
print()
print("LEAD:   ", article.get("vorspann_ctx", "")[:300])
print()
body_plain = strip_html(article.get("grundtext_ctx", ""))
print("BODY:   ", body_plain[:400], "...")
print()
print("AUTOR:  ", article.get("autor_ctx", ""))
print("RESSORT:", article.get("lt_ressort_name", ""), "/", article.get("lt_unterressort_name", ""))
print("DATUM:  ", (article.get("published_from_ts") or [""])[0][:10])
print("ID:     ", article.get("media_id"))
print("ZEICHEN:", article.get("zeichenanzahl", 0))

## 3. Datensatz-Überblick
Alle Monatsdateien laden und aggregierte Statistiken berechnen.

In [ ]:
# Alle Dateien: Artikel pro Monat
files = sorted(glob.glob(NZZ_JSON_GLOB))
rows = []
for path in files:
    with open(path) as f:
        data = json.load(f)
    wrapper = data[0] if isinstance(data, list) else data
    month = path.split("/")[-1].replace("articles_", "").replace(".json", "")
    n = len(wrapper.get("result", []))
    rows.append({"monat": month, "artikel": n})

monthly = pd.DataFrame(rows)
print(f"Dateien gesamt:  {len(files)}")
print(f"Artikel gesamt:  {monthly['artikel'].sum():,}")
print()
print(monthly.to_string(index=False))

In [ ]:
monthly.plot(x="monat", y="artikel", kind="bar", figsize=(14, 4), legend=False, color="steelblue")
plt.title("Artikel pro Monat")
plt.xlabel("")
plt.ylabel("Anzahl Artikel")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4. Preprocessing testen
`load_dataset()` + `preprocess()` auf den NZZ-Daten — HTML-Stripping, Filterung leerer Artikel, Textlängen.

In [ ]:
raw_df = load_dataset()
print(f"Rohdaten:  {len(raw_df):,} Artikel")
print(f"Spalten:   {list(raw_df.columns)}")
print()

# Wie viele haben keinen Body (Multimedia etc.)?
empty = raw_df[raw_df["body"].str.strip() == ""]
print(f"Leere Bodies (Multimedia-Artikel): {len(empty):,}")

df = preprocess(raw_df)
print(f"Nach Preprocessing: {len(df):,} Artikel (gefiltert: {len(raw_df)-len(df):,})")
print()
print("Ressort-Verteilung:")
print(df["category"].value_counts().head(15).to_string())

In [ ]:
# Ressort-Verteilung als Balkendiagramm
df["category"].value_counts().head(20).plot(
    kind="barh", figsize=(9, 6), color="steelblue"
)
plt.title("Artikel pro Ressort (Top 20)")
plt.xlabel("Anzahl Artikel")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Body-Länge Verteilung
df["body_len"] = df["body"].str.len()
print(df["body_len"].describe().round(0).to_string())

df["body_len"].clip(upper=10000).hist(bins=50, figsize=(10, 4), color="steelblue", edgecolor="white")
plt.title("Verteilung der Body-Länge (Zeichen, max. 10k gezeigt)")
plt.xlabel("Zeichen")
plt.ylabel("Artikel")
plt.tight_layout()
plt.show()

In [ ]:
# Beispielartikel anzeigen
sample = df[df["body_len"] > 2000].iloc[0]
print(f"Titel:    {sample['title']}")
print(f"Autor:    {sample['author']}")
print(f"Ressort:  {sample['category']} / {sample.get('subcategory', '')}")
print(f"Datum:    {sample['published_date']}")
print(f"Zeichen:  {sample['body_len']}")
print()
print("Lead:")
print(sample["lead"])
print()
print("Body (erste 600 Zeichen):")
print(sample["body"][:600], "...")

## 5. Chunking testen

In [ ]:
chunks = chunk_dataframe(df)

print(f"Artikel:               {len(df):,}")
print(f"Chunks gesamt:         {len(chunks):,}")
print(f"Ø Chunks pro Artikel:  {len(chunks)/len(df):.1f}")
print()
chunks["chunk_len"] = chunks["chunk_text"].str.len()
print(chunks["chunk_len"].describe().round(0).to_string())
print()
print("Beispiel-Chunk:")
print(chunks.iloc[1].to_string())

## 6. Sample-Embedding & Retrieval testen
Kleines Sample (3 Artikel pro Ressort) in eine temporäre ChromaDB-Collection einbetten und Retrieval testen.  
**Modell wird hier erst geladen** — dauert 1–2 Minuten.

In [ ]:
from embed import load_model, embed_chunks, get_chroma_collection

model = load_model(EMBEDDING_MODEL)
print("Modell geladen.")

In [ ]:
# 3 Artikel pro Ressort samplen, chunken, embedden
sample_df     = df.groupby("category").head(3).reset_index(drop=True)
sample_chunks = chunk_dataframe(sample_df)
print(f"Sample: {len(sample_df)} Artikel, {len(sample_chunks)} Chunks")

embeddings = embed_chunks(model, sample_chunks["chunk_text"].tolist())

# Temp-Collection (in-memory, kein Schreiben auf Disk)
import chromadb
tmp_client     = chromadb.EphemeralClient()
tmp_collection = tmp_client.get_or_create_collection("explorer_test", metadata={"hnsw:space": "cosine"})

meta_cols = ["article_id", "chunk_index", "title", "category", "author", "published_date"]
tmp_collection.upsert(
    ids        = sample_chunks["chunk_id"].tolist(),
    embeddings = embeddings,
    documents  = sample_chunks["chunk_text"].tolist(),
    metadatas  = sample_chunks[meta_cols].to_dict(orient="records"),
)
print("Sample in temporäre ChromaDB geladen.")

In [ ]:
from retrieval import embed_query, search

# NZZ-spezifische Testqueries
queries = [
    "Wie hat die Schweiz auf den Ukraine-Krieg reagiert?",
    "SNB Zinsentscheid Franken",
    "Bundesratswahl Nachfolge",
    "KI künstliche Intelligenz Regulierung Europa",
    "Klimawandel Gletscher Schweiz",
]

for query in queries:
    q_emb   = embed_query(model, query)
    results = search(tmp_collection, q_emb, top_k=3)
    print(f"Query: {query}")
    for r in results:
        author = f" — {r['author']}" if r.get("author") else ""
        print(f"  [{r['category']}] {r['title'][:65]}{author}  ({r['similarity_score']:.3f})")
    print()

## 7. Evaluation gegen Ground Truth

Retrieval auf der echten ChromaDB (nicht Sample) gegen die definierten Ground-Truth-Queries messen.  
Metriken: **Hit@1**, **Hit@3**, **Hit@5** — trifft der richtige Artikel unter den Top-K Ergebnissen?

In [ ]:
from retrieval import load_models, retrieve

# Modell laden falls noch nicht aus Sektion 6 vorhanden
try:
    model
except NameError:
    model = None

if model is None:
    model, reranker = load_models(use_reranking=USE_RERANKING)
else:
    from sentence_transformers import CrossEncoder
    from config import RERANKER_MODEL
    reranker = CrossEncoder(RERANKER_MODEL) if USE_RERANKING else None
    print("Modell aus Sektion 6 wiederverwendet.")

collection = get_chroma_collection(CHROMA_PATH, CHROMA_COLLECTION)
print(f"ChromaDB verbunden: {collection.count():,} Chunks")
print(f"Reranking aktiv:    {USE_RERANKING}")

In [ ]:
from config import EVAL_GROUND_TRUTH, EVAL_TOP_K_RETRIEVAL, EVAL_TOP_K_FINAL

# Ground Truth laden
with open(EVAL_GROUND_TRUTH) as f:
    ground_truth = [json.loads(line) for line in f if line.strip()]

print(f"Ground-Truth-Queries: {len(ground_truth)}")

# Retrieval für jede Query ausführen
TOP_K = EVAL_TOP_K_FINAL   # für Hit@k Berechnung

eval_rows = []
for sample in ground_truth:
    query        = sample["query"]
    relevant_ids = set(sample["relevant_article_ids"])

    results = retrieve(
        query, model, collection, reranker,
        top_k_retrieval=EVAL_TOP_K_RETRIEVAL,
        top_k_rerank=TOP_K,
    )

    returned_ids = [r["article_id"] for r in results]

    hit1 = any(rid in relevant_ids for rid in returned_ids[:1])
    hit3 = any(rid in relevant_ids for rid in returned_ids[:3])
    hit5 = any(rid in relevant_ids for rid in returned_ids[:5])

    best = results[0] if results else {}
    eval_rows.append({
        "query":          query,
        "hit@1":          hit1,
        "hit@3":          hit3,
        "hit@5":          hit5,
        "best_score":     round(best.get("similarity_score", 0), 3),
        "best_title":     best.get("title", "")[:60],
        "best_category":  best.get("category", ""),
        "relevant_ids":   list(relevant_ids),
        "returned_ids":   returned_ids,
        "results":        results,
    })

eval_df = pd.DataFrame(eval_rows)
print("Fertig.")
print(f"\nHit@1: {eval_df['hit@1'].mean():.0%}   Hit@3: {eval_df['hit@3'].mean():.0%}   Hit@5: {eval_df['hit@5'].mean():.0%}")

In [ ]:
# ── Übersichts-Tabelle ──────────────────────────────────────────────────────
def style_hit(val):
    return "background-color: #c6efce; color: #276221" if val else "background-color: #ffc7ce; color: #9c0006"

display_df = eval_df[["query", "hit@1", "hit@3", "hit@5", "best_score", "best_title", "best_category"]].copy()
display_df["query"] = display_df["query"].str[:65]

(display_df.style
    .applymap(style_hit, subset=["hit@1", "hit@3", "hit@5"])
    .format({"best_score": "{:.3f}"})
    .set_properties(**{"text-align": "left"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
)

In [ ]:
# ── Metriken-Balkendiagramm ─────────────────────────────────────────────────
metrics = {
    "Hit@1": eval_df["hit@1"].mean(),
    "Hit@3": eval_df["hit@3"].mean(),
    "Hit@5": eval_df["hit@5"].mean(),
}
colors = ["#2196F3", "#4CAF50", "#8BC34A"]

fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(metrics.keys(), metrics.values(), color=colors, width=0.4)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Hit Rate")
ax.set_title(f"Retrieval Metriken — {len(eval_df)} Queries")
ax.axhline(1.0, color="gray", linewidth=0.8, linestyle="--")
for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{val:.0%}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Detail-Ansicht: Top-5 Ergebnisse pro Query ──────────────────────────────
from IPython.display import display, HTML

for row in eval_rows:
    relevant = set(row["relevant_ids"])
    icon     = "✅" if row["hit@5"] else "❌"
    html     = f"<hr><b>{icon} {row['query']}</b><br>"
    html    += f"<small>Hit@1={row['hit@1']} &nbsp; Hit@3={row['hit@3']} &nbsp; Hit@5={row['hit@5']}</small>"
    html    += "<table style='width:100%; font-size:13px; margin-top:6px'>"
    html    += "<tr><th>#</th><th>Score</th><th>Artikel-ID</th><th>Ressort</th><th>Titel</th><th>Autor</th><th>Datum</th></tr>"
    for rank, r in enumerate(row["results"], 1):
        is_hit  = r["article_id"] in relevant
        bg      = "#c6efce" if is_hit else "white"
        marker  = " ✓" if is_hit else ""
        html   += (
            f"<tr style='background:{bg}'>"
            f"<td>{rank}</td>"
            f"<td>{r['similarity_score']:.3f}</td>"
            f"<td>{r['article_id']}{marker}</td>"
            f"<td>{r['category']}</td>"
            f"<td>{r['title'][:70]}</td>"
            f"<td>{r.get('author','')[:25]}</td>"
            f"<td>{r.get('published_date','')}</td>"
            f"</tr>"
        )
    html += "</table>"
    display(HTML(html))